# Geospatial Delivery Analysis & Route Optimization


---

## Context

A logistics company operating in Bogota wants to **optimize its delivery routes** and **identify coverage gaps** across the city. The company operates three warehouses and serves thousands of deliveries daily.

### Objectives

1. Understand delivery density patterns and spatial distribution
2. Identify delivery hotspots and natural zone clusters
3. Predict delivery times based on distance, time, and zone features
4. Optimize warehouse-to-delivery assignments to reduce total distance
5. Detect underserved areas and recommend new warehouse locations

### Tools & Approach

We use spatial clustering (K-Means, DBSCAN), regression models (Linear Regression, Random Forest), and coverage density analysis -- all with standard Python libraries (no folium/geopandas required).

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

SEED = 42
np.random.seed(SEED)

print('All imports loaded successfully.')

---
## 2. Data Loading & Preparation

We generate ~5,000 deliveries with coordinates resembling Bogota's geography (lat 4.60-4.75, lon -74.15 to -74.0), three warehouse origins, and realistic delivery attributes.

In [ ]:
N = 5000

# --- Warehouse locations (fixed) ---
warehouses = pd.DataFrame({
    'warehouse_id': ['WH_Norte', 'WH_Centro', 'WH_Sur'],
    'lat': [4.72, 4.65, 4.61],
    'lon': [-74.05, -74.08, -74.12]
})

# --- Zone centers (5 zones) ---
zone_centers = {
    'Usaquen':   (4.73, -74.03),
    'Chapinero': (4.70, -74.06),
    'Centro':    (4.65, -74.08),
    'Kennedy':   (4.62, -74.13),
    'Suba':      (4.74, -74.10),
}

zones = list(zone_centers.keys())
zone_probs = [0.20, 0.25, 0.22, 0.18, 0.15]

# Assign each delivery to a zone
zone_assignments = np.random.choice(zones, size=N, p=zone_probs)

# Generate coordinates around zone centers with jitter
lats, lons = [], []
for z in zone_assignments:
    clat, clon = zone_centers[z]
    lats.append(clat + np.random.normal(0, 0.015))
    lons.append(clon + np.random.normal(0, 0.015))

lats = np.clip(lats, 4.58, 4.78)
lons = np.clip(lons, -74.18, -73.98)

# Assign warehouse origin (weighted by proximity)
warehouse_origin = np.random.choice(
    warehouses['warehouse_id'].values, size=N, p=[0.35, 0.40, 0.25]
)

# Time and day features
hour_of_day = np.random.choice(range(7, 22), size=N, p=[
    0.03, 0.06, 0.09, 0.11, 0.10, 0.10, 0.09, 0.08,
    0.07, 0.06, 0.06, 0.05, 0.04, 0.03, 0.03
])
day_of_week = np.random.choice(
    ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    size=N, p=[0.17, 0.16, 0.16, 0.15, 0.15, 0.12, 0.09]
)

# Distance (km) -- derived from warehouse to delivery point + noise
wh_lat_map = dict(zip(warehouses['warehouse_id'], warehouses['lat']))
wh_lon_map = dict(zip(warehouses['warehouse_id'], warehouses['lon']))

distance_km = []
for i in range(N):
    dlat = lats[i] - wh_lat_map[warehouse_origin[i]]
    dlon = lons[i] - wh_lon_map[warehouse_origin[i]]
    raw_dist = np.sqrt(dlat**2 + dlon**2) * 111  # rough degree-to-km
    distance_km.append(max(0.5, raw_dist + np.random.normal(0, 0.5)))

distance_km = np.array(distance_km)

# Delivery time (minutes) -- correlated with distance + rush-hour effect
rush_factor = np.where(np.isin(hour_of_day, [8, 9, 12, 13, 17, 18]), 1.3, 1.0)
weekend_factor = np.where(np.isin(day_of_week, ['Sat', 'Sun']), 0.85, 1.0)
delivery_time_minutes = (
    10 + distance_km * 3.5 * rush_factor * weekend_factor
    + np.random.normal(0, 5, N)
).clip(5, 120).round(1)

# Delivery status
delay_prob = 0.12 + 0.08 * (rush_factor - 1) + 0.03 * (distance_km / distance_km.max())
delivery_status = np.where(np.random.random(N) < delay_prob, 'delayed', 'on_time')

# Order value
order_value = (np.random.lognormal(mean=10.5, sigma=0.6, size=N)).round(0).astype(int)
order_value = np.clip(order_value, 5_000, 500_000)

# Build DataFrame
df = pd.DataFrame({
    'delivery_id': [f'DEL-{i:05d}' for i in range(N)],
    'lat': np.round(lats, 6),
    'lon': np.round(lons, 6),
    'zone': zone_assignments,
    'warehouse_origin': warehouse_origin,
    'hour_of_day': hour_of_day,
    'day_of_week': day_of_week,
    'distance_km': np.round(distance_km, 2),
    'delivery_time_minutes': delivery_time_minutes,
    'order_value_cop': order_value,
    'delivery_status': delivery_status,
})

print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
df.info()
print()
df.describe()

---
## 3. Exploratory Data Analysis

In [ ]:
# 3.1  Delivery density scatter plot (lat/lon colored by zone)
fig, ax = plt.subplots(figsize=(9, 8))
zone_colors = {'Usaquen': '#e74c3c', 'Chapinero': '#3498db',
               'Centro': '#2ecc71', 'Kennedy': '#f39c12', 'Suba': '#9b59b6'}

for zone, color in zone_colors.items():
    mask = df['zone'] == zone
    ax.scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
               s=6, alpha=0.4, label=zone, color=color)

# Plot warehouses
ax.scatter(warehouses['lon'], warehouses['lat'],
           marker='^', s=200, c='black', zorder=5, edgecolors='white', linewidths=1.5)
for _, wh in warehouses.iterrows():
    ax.annotate(wh['warehouse_id'], (wh['lon'], wh['lat']),
                fontsize=8, fontweight='bold', ha='left',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Delivery Density by Zone (Bogota)', fontsize=13, fontweight='bold')
ax.legend(title='Zone', markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2  Delivery time distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(df['delivery_time_minutes'], bins=50, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(df['delivery_time_minutes'].median(), color='red', ls='--', label=f"Median: {df['delivery_time_minutes'].median():.1f} min")
axes[0].set_xlabel('Delivery Time (min)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Delivery Times')
axes[0].legend()

# On-time rate by zone
ontime_rate = df.groupby('zone')['delivery_status'].apply(
    lambda x: (x == 'on_time').mean() * 100
).sort_values()

bars = axes[1].barh(ontime_rate.index, ontime_rate.values, color='#2ecc71', edgecolor='white')
axes[1].set_xlabel('On-Time Rate (%)')
axes[1].set_title('On-Time Delivery Rate by Zone')
axes[1].set_xlim(70, 100)
for bar, val in zip(bars, ontime_rate.values):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 3.3  Heatmap: deliveries by hour and day
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot = df.groupby(['day_of_week', 'hour_of_day']).size().unstack(fill_value=0)
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='d', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Deliveries'})
ax.set_title('Delivery Volume by Hour & Day of Week', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4  Delay rate analysis by hour
delay_by_hour = df.groupby('hour_of_day')['delivery_status'].apply(
    lambda x: (x == 'delayed').mean() * 100
).reset_index(name='delay_pct')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(delay_by_hour['hour_of_day'], delay_by_hour['delay_pct'],
       color='#e74c3c', edgecolor='white', alpha=0.85)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Delay Rate (%)')
ax.set_title('Delivery Delay Rate by Hour', fontsize=13, fontweight='bold')
ax.set_xticks(range(7, 22))
plt.tight_layout()
plt.show()

---
## 4. Spatial Clustering

### 4.1 K-Means: Identifying Natural Delivery Zones

We use K-Means to let the algorithm discover delivery zones from coordinates alone, then compare with the manually labeled zones.

In [ ]:
coords = df[['lat', 'lon']].values

# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(coords)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_range, inertias, 'o-', color='#3498db', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('K-Means Elbow Method', fontweight='bold')
ax.axvline(5, color='red', ls='--', alpha=0.6, label='k=5 (selected)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with k=5
kmeans = KMeans(n_clusters=5, random_state=SEED, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(coords)
centers = kmeans.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Original zones
for zone, color in zone_colors.items():
    mask = df['zone'] == zone
    axes[0].scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                    s=6, alpha=0.4, label=zone, color=color)
axes[0].set_title('Original Labeled Zones', fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=3)

# K-Means clusters
scatter = axes[1].scatter(df['lon'], df['lat'], c=df['kmeans_cluster'],
                          cmap='Set1', s=6, alpha=0.4)
axes[1].scatter(centers[:, 1], centers[:, 0], marker='X', s=200,
                c='black', edgecolors='white', linewidths=1.5, zorder=5)
axes[1].set_title('K-Means Clusters (k=5)', fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

print('Cluster sizes:')
print(df['kmeans_cluster'].value_counts().sort_index())

### 4.2 DBSCAN: Hotspot Detection

DBSCAN identifies dense regions (hotspots) without requiring a pre-specified number of clusters. Points not belonging to any dense region are marked as noise (-1).

In [ ]:
dbscan = DBSCAN(eps=0.008, min_samples=30)
df['dbscan_cluster'] = dbscan.fit_predict(coords)

n_hotspots = df['dbscan_cluster'].nunique() - (1 if -1 in df['dbscan_cluster'].values else 0)
noise_pct = (df['dbscan_cluster'] == -1).mean() * 100

fig, ax = plt.subplots(figsize=(9, 8))
noise_mask = df['dbscan_cluster'] == -1
ax.scatter(df.loc[noise_mask, 'lon'], df.loc[noise_mask, 'lat'],
           s=4, alpha=0.15, color='gray', label='Noise')

cmap_vals = plt.cm.tab10(np.linspace(0, 1, n_hotspots))
for idx, cluster_id in enumerate(sorted(df.loc[~noise_mask, 'dbscan_cluster'].unique())):
    mask = df['dbscan_cluster'] == cluster_id
    ax.scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
               s=8, alpha=0.5, color=cmap_vals[idx], label=f'Hotspot {cluster_id}')

ax.scatter(warehouses['lon'], warehouses['lat'],
           marker='^', s=200, c='black', zorder=5, edgecolors='white', linewidths=1.5)
ax.set_title(f'DBSCAN Hotspot Detection ({n_hotspots} hotspots, {noise_pct:.1f}% noise)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

---
## 5. Delivery Time Prediction

We build regression models to predict `delivery_time_minutes` using distance, time of day, day of week, and zone.

In [ ]:
# Feature engineering
le_zone = LabelEncoder()
le_day = LabelEncoder()

df_model = df.copy()
df_model['zone_enc'] = le_zone.fit_transform(df_model['zone'])
df_model['day_enc'] = le_day.fit_transform(df_model['day_of_week'])
df_model['is_rush'] = df_model['hour_of_day'].isin([8, 9, 12, 13, 17, 18]).astype(int)
df_model['is_weekend'] = df_model['day_of_week'].isin(['Sat', 'Sun']).astype(int)

features = ['distance_km', 'hour_of_day', 'day_enc', 'zone_enc', 'is_rush', 'is_weekend']
target = 'delivery_time_minutes'

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
# Train models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=SEED)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'MAE': mae, 'RMSE': rmse, 'preds': y_pred}
    print(f'{name:25s}  MAE: {mae:.2f} min   RMSE: {rmse:.2f} min')

# Results table
results_df = pd.DataFrame({k: {m: v for m, v in vals.items() if m != 'preds'}
                           for k, vals in results.items()}).T
results_df

In [ ]:
# Actual vs Predicted scatter (Random Forest)
best_preds = results['Random Forest']['preds']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, best_preds, s=8, alpha=0.3, color='#3498db')
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Delivery Time (min)')
axes[0].set_ylabel('Predicted Delivery Time (min)')
axes[0].set_title('Random Forest: Actual vs Predicted', fontweight='bold')
axes[0].legend()

# Feature importance
rf_model = models['Random Forest']
importance = pd.Series(rf_model.feature_importances_, index=features).sort_values()
importance.plot.barh(ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## 6. Zone Optimization: Warehouse Assignment

Currently, deliveries are randomly assigned to warehouses. We reassign each delivery to its **nearest warehouse** and measure the distance savings.

In [ ]:
# Calculate distance from each delivery to each warehouse
delivery_coords = df[['lat', 'lon']].values
wh_coords = warehouses[['lat', 'lon']].values

# Distance matrix (Euclidean in degrees * ~111 km/deg)
dist_matrix = cdist(delivery_coords, wh_coords) * 111  # approximate km

# Current assignment distances
wh_id_to_idx = {wid: i for i, wid in enumerate(warehouses['warehouse_id'])}
current_wh_idx = df['warehouse_origin'].map(wh_id_to_idx).values
current_distances = dist_matrix[np.arange(N), current_wh_idx]

# Optimized: assign to nearest warehouse
optimal_wh_idx = dist_matrix.argmin(axis=1)
optimal_distances = dist_matrix[np.arange(N), optimal_wh_idx]
optimal_wh_names = warehouses['warehouse_id'].values[optimal_wh_idx]

df['optimized_warehouse'] = optimal_wh_names
df['current_wh_distance_km'] = np.round(current_distances, 2)
df['optimal_wh_distance_km'] = np.round(optimal_distances, 2)
df['distance_saved_km'] = np.round(current_distances - optimal_distances, 2)

total_current = current_distances.sum()
total_optimal = optimal_distances.sum()
total_saved = total_current - total_optimal
pct_saved = (total_saved / total_current) * 100

print(f'Total current distance:   {total_current:,.1f} km')
print(f'Total optimized distance: {total_optimal:,.1f} km')
print(f'Total distance saved:     {total_saved:,.1f} km ({pct_saved:.1f}%)')
print(f'\nReassigned deliveries:    {(df["warehouse_origin"] != df["optimized_warehouse"]).sum()} / {N}')

In [ ]:
# Visualize current vs optimized assignments
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
wh_colors = {'WH_Norte': '#e74c3c', 'WH_Centro': '#3498db', 'WH_Sur': '#2ecc71'}

for label, ax_idx, col in [('Current Assignment', 0, 'warehouse_origin'),
                            ('Optimized (Nearest)', 1, 'optimized_warehouse')]:
    ax = axes[ax_idx]
    for wh, color in wh_colors.items():
        mask = df[col] == wh
        ax.scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                   s=5, alpha=0.3, color=color, label=wh)
    ax.scatter(warehouses['lon'], warehouses['lat'],
               marker='^', s=200, c='black', zorder=5, edgecolors='white', linewidths=1.5)
    ax.set_title(label, fontweight='bold', fontsize=12)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.legend(markerscale=3, fontsize=8)

plt.suptitle(f'Warehouse Assignment Optimization (saves {pct_saved:.1f}% distance)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Coverage Analysis

We identify **underserved areas** -- regions with low delivery density that might benefit from a new warehouse.

In [ ]:
# Grid-based density analysis
lat_bins = np.linspace(df['lat'].min(), df['lat'].max(), 25)
lon_bins = np.linspace(df['lon'].min(), df['lon'].max(), 25)

density, lat_edges, lon_edges = np.histogram2d(df['lat'], df['lon'],
                                                bins=[lat_bins, lon_bins])

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.pcolormesh(lon_edges, lat_edges, density, cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='Delivery Count')

ax.scatter(warehouses['lon'], warehouses['lat'],
           marker='^', s=200, c='blue', zorder=5, edgecolors='white', linewidths=1.5)
for _, wh in warehouses.iterrows():
    ax.annotate(wh['warehouse_id'], (wh['lon'], wh['lat']),
                fontsize=8, fontweight='bold', color='blue',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Delivery Density Grid (Coverage Map)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Identify underserved areas: deliveries far from any warehouse
min_wh_dist = dist_matrix.min(axis=1)
threshold_km = np.percentile(min_wh_dist, 85)  # top 15% most distant

underserved_mask = min_wh_dist > threshold_km
underserved = df[underserved_mask]

# Suggest new warehouse location: centroid of underserved cluster
suggested_lat = underserved['lat'].mean()
suggested_lon = underserved['lon'].mean()

print(f'Underserved deliveries: {underserved_mask.sum()} ({underserved_mask.mean()*100:.1f}%)')
print(f'Distance threshold: {threshold_km:.2f} km')
print(f'\nSuggested new warehouse location:')
print(f'  Latitude:  {suggested_lat:.6f}')
print(f'  Longitude: {suggested_lon:.6f}')

# Visualize
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(df.loc[~underserved_mask, 'lon'], df.loc[~underserved_mask, 'lat'],
           s=5, alpha=0.2, color='#3498db', label='Well-served')
ax.scatter(underserved['lon'], underserved['lat'],
           s=10, alpha=0.5, color='#e74c3c', label='Underserved')

# Current warehouses
ax.scatter(warehouses['lon'], warehouses['lat'],
           marker='^', s=200, c='black', zorder=5, edgecolors='white', linewidths=1.5,
           label='Current Warehouse')

# Suggested new warehouse
ax.scatter(suggested_lon, suggested_lat,
           marker='*', s=400, c='gold', zorder=5, edgecolors='black', linewidths=1.5,
           label='Suggested New Warehouse')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Coverage Gap Analysis & Suggested Warehouse Location',
             fontsize=12, fontweight='bold')
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.show()

---
## 8. KPI Dashboard

A summary dashboard with key logistics performance indicators.

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# --- KPI 1: Avg delivery time by zone ---
ax1 = fig.add_subplot(gs[0, 0])
avg_time = df.groupby('zone')['delivery_time_minutes'].mean().sort_values()
bars1 = ax1.barh(avg_time.index, avg_time.values, color='#3498db', edgecolor='white')
ax1.set_xlabel('Avg Delivery Time (min)')
ax1.set_title('Avg Delivery Time by Zone', fontweight='bold', fontsize=11)
for bar, val in zip(bars1, avg_time.values):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=9)

# --- KPI 2: On-time % by zone ---
ax2 = fig.add_subplot(gs[0, 1])
ontime = df.groupby('zone')['delivery_status'].apply(
    lambda x: (x == 'on_time').mean() * 100
).sort_values()
colors_ot = ['#e74c3c' if v < 85 else '#2ecc71' for v in ontime.values]
bars2 = ax2.barh(ontime.index, ontime.values, color=colors_ot, edgecolor='white')
ax2.set_xlabel('On-Time Rate (%)')
ax2.set_title('On-Time Rate by Zone', fontweight='bold', fontsize=11)
ax2.set_xlim(70, 100)
for bar, val in zip(bars2, ontime.values):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

# --- KPI 3: Deliveries per hour ---
ax3 = fig.add_subplot(gs[0, 2])
hourly = df.groupby('hour_of_day').size()
ax3.fill_between(hourly.index, hourly.values, alpha=0.3, color='#9b59b6')
ax3.plot(hourly.index, hourly.values, 'o-', color='#9b59b6', linewidth=2, markersize=5)
ax3.set_xlabel('Hour of Day')
ax3.set_ylabel('Deliveries')
ax3.set_title('Deliveries per Hour', fontweight='bold', fontsize=11)
ax3.set_xticks(range(7, 22))

# --- KPI 4: Distance distribution ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(df['distance_km'], bins=40, color='#f39c12', edgecolor='white', alpha=0.85)
ax4.axvline(df['distance_km'].median(), color='red', ls='--',
            label=f"Median: {df['distance_km'].median():.1f} km")
ax4.set_xlabel('Distance (km)')
ax4.set_ylabel('Count')
ax4.set_title('Delivery Distance Distribution', fontweight='bold', fontsize=11)
ax4.legend(fontsize=9)

# --- KPI 5: Warehouse load (current vs optimized) ---
ax5 = fig.add_subplot(gs[1, 1])
current_load = df['warehouse_origin'].value_counts().sort_index()
optimal_load = df['optimized_warehouse'].value_counts().sort_index()
x_pos = np.arange(len(current_load))
width = 0.35
ax5.bar(x_pos - width/2, current_load.values, width, label='Current', color='#e74c3c', alpha=0.8)
ax5.bar(x_pos + width/2, optimal_load.values, width, label='Optimized', color='#2ecc71', alpha=0.8)
ax5.set_xticks(x_pos)
ax5.set_xticklabels(current_load.index, fontsize=9)
ax5.set_ylabel('Deliveries')
ax5.set_title('Warehouse Load Distribution', fontweight='bold', fontsize=11)
ax5.legend(fontsize=9)

# --- KPI 6: Avg order value by zone ---
ax6 = fig.add_subplot(gs[1, 2])
avg_order = df.groupby('zone')['order_value_cop'].mean().sort_values() / 1000
bars6 = ax6.barh(avg_order.index, avg_order.values, color='#1abc9c', edgecolor='white')
ax6.set_xlabel('Avg Order Value (COP thousands)')
ax6.set_title('Avg Order Value by Zone', fontweight='bold', fontsize=11)
for bar, val in zip(bars6, avg_order.values):
    ax6.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'${val:.0f}K', va='center', fontsize=9)

fig.suptitle('Logistics KPI Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Conclusions & Recommendations

### Key Findings

1. **Spatial clustering** (K-Means with k=5) closely matches the manually labeled delivery zones, validating the zone definitions. DBSCAN revealed dense delivery hotspots within each zone.

2. **Delivery time prediction**: The Random Forest model significantly outperforms Linear Regression, achieving a lower MAE and RMSE. **Distance** is the dominant predictor, followed by rush-hour effects.

3. **Warehouse optimization**: Reassigning deliveries to the nearest warehouse reduces total travel distance by a meaningful percentage. This translates to fuel savings, faster delivery times, and lower operational costs.

4. **Coverage gaps**: Approximately 15% of deliveries are significantly far from any warehouse. The suggested new warehouse location (centroid of underserved deliveries) would improve coverage in peripheral areas.

### Recommendations

| Priority | Action | Expected Impact |
|----------|--------|-----------------|
| High | Implement nearest-warehouse assignment | Immediate distance/fuel savings |
| High | Adjust staffing for rush hours (8-9, 12-13, 17-18) | Reduce delay rate during peak times |
| Medium | Evaluate a 4th warehouse at the suggested location | Improve coverage for underserved areas |
| Medium | Use the Random Forest model for delivery time estimates | Better customer communication and route planning |
| Low | Refine zone boundaries using K-Means clusters | Align operational zones with actual delivery patterns |

### Next Steps

- Incorporate real traffic data and road network distances (e.g., OSRM) for more accurate routing
- Build a time-series model to forecast daily delivery volumes by zone
- Run a cost-benefit analysis for the proposed 4th warehouse